# 5-тапсырма: SMS деректерінің EDA

Алдымен `python -m ml.data.load_sms` және `python -m ml.preprocess_sms` орындаңыз. Ноутбук `data/processed/sms_clean.csv` оқиды. Графиктер `docs/figures/` ішіне сақталады.

In [ ]:
import sys, re
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

FIG = ROOT / "docs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(ROOT / "data" / "processed" / "sms_clean.csv", encoding="utf-8", keep_default_na=False)
df["name"] = df.label.map({0: "ham", 1: "spam"})
df.head()

## 1. Класс таралуы

In [ ]:
vc = df.name.value_counts()
ax = vc.plot(kind="bar", color=["#198754", "#dc3545"], rot=0)
for i, v in enumerate(vc):
    ax.text(i, v, f"{v} ({v/len(df)*100:.1f}%)", ha="center", va="bottom")
ax.set_title("Распределение классов"); ax.set_xlabel("Класс"); ax.set_ylabel("Число сообщений")
plt.tight_layout(); plt.savefig(FIG / "sms_class_balance.png", dpi=150); plt.show()

## 2. Хабарлама ұзындығы

In [ ]:
df["len_chars"] = df.text_raw.str.len()
df["len_words"] = df.text_raw.str.split().str.len()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.boxplot(column="len_chars", by="name", ax=axes[0]); axes[0].set_title("Длина в символах"); axes[0].set_xlabel("Класс"); axes[0].set_ylabel("Символов")
df.boxplot(column="len_words", by="name", ax=axes[1]); axes[1].set_title("Длина в словах"); axes[1].set_xlabel("Класс"); axes[1].set_ylabel("Слов")
plt.suptitle(""); plt.tight_layout(); plt.savefig(FIG / "sms_length.png", dpi=150); plt.show()
df.groupby("name")[["len_chars", "len_words"]].mean().round(1)

## 3. Топ-20 сөз және биграмма (стоп-сөзсіз)

In [ ]:
def top_ngrams(texts, ngram, k=20):
    cv = CountVectorizer(ngram_range=ngram, stop_words="english", token_pattern=r"(?u)\b[^\W\d_]{2,}\b")
    X = cv.fit_transform(texts)
    s = pd.Series(X.sum(axis=0).A1, index=cv.get_feature_names_out())
    return s.sort_values(ascending=False).head(k)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for r, (lab, name) in enumerate([(0, "ham"), (1, "spam")]):
    for c, (ng, title) in enumerate([((1, 1), "слова"), ((2, 2), "биграммы")]):
        s = top_ngrams(df[df.label == lab].text_clean, ng)
        s[::-1].plot(kind="barh", ax=axes[r][c], color="#198754" if lab == 0 else "#dc3545")
        axes[r][c].set_title(f"Топ-20 {title}: {name}"); axes[r][c].set_xlabel("Частота")
plt.tight_layout(); plt.savefig(FIG / "sms_top_ngrams.png", dpi=150); plt.show()

## 4. Токендер мен «!» үлесі

In [ ]:
feat = pd.DataFrame({
    "<url>":   df.text_clean.str.contains("<url>", regex=False),
    "<phone>": df.text_clean.str.contains("<phone>", regex=False),
    "<money>": df.text_clean.str.contains("<money>", regex=False),
    "!":       df.text_clean.str.contains("!", regex=False),
})
share = (feat.groupby(df.name).mean() * 100).round(1)
ax = share.T.plot(kind="bar", rot=0, color=["#198754", "#dc3545"])
ax.set_title("Доля сообщений с признаком по классам"); ax.set_xlabel("Признак"); ax.set_ylabel("Доля, %")
plt.tight_layout(); plt.savefig(FIG / "sms_token_share.png", dpi=150); plt.show()
share

## 5. Қорытындылар (кемінде 5, әрқайсысы санмен)

Жоғарыдағы нәтижелерге қарап, **өз сандарыңызбен** толтырыңыз. Мысалдар:

1. Spam хабарламалары орта есеппен ұзынырақ: ham ___ символ, spam ___ символ.
2. `<phone>` spam-да ___% , ham-да ___% кездеседі.
3. `<money>` белгісі spam-да ___% .
4. «!» spam-да ___% , ham-да ___% .
5. Класс дисбалансы: ham ___%, spam ___%.

**Дисбалансты қалай ескереміз:** стратификацияланған бөлу, `class_weight='balanced'`, Recall/F2 метрикасы.